# Transfer Learning from MNIST to EMNIST Letters Using a Custom CNN

This notebook demonstrates the complete transfer learning workflow using a custom convolutional neural network (CNN).

We will:

1. Train a CNN on the MNIST handwritten digit dataset
2. Save the pretrained weights
3. Transfer the learned features to EMNIST letter classification
4. Freeze the convolutional backbone
5. Fine-tune the model on the new task
6. Evaluate performance using multiple metrics

The goal is to understand:
- feature learning
- transfer learning
- freezing vs fine-tuning
- domain adaptation


### Why MNIST Features Transfer to EMNIST

The first convolutional layers learn very generic visual primitives such as:

- edges,
- corners,
- curves,
- and stroke transitions.

These structures appear in both:

- handwritten digits,
- and handwritten letters.

Because the low-level visual statistics are similar, the pretrained filters learned on MNIST can accelerate learning on EMNIST.

Only the higher-level classifier must adapt to the new semantic categories.

> Although EMNIST letters share visual similarities with MNIST digits (strokes, curves, edges), the distributions are not identical.<br>Fine-tuning allows the pretrained convolutional filters to adapt to these new structures.


## Imports and Environment Setup

This notebook uses:
- PyTorch for deep learning and GPU acceleration,
- torchvision for datasets and image preprocessing,
- scikit-learn for evaluation metrics,
- and matplotlib for visualization.

Additional utilities are used for:
- reproducibility,
- dataset inspection,
- and training progress monitoring.


In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from tqdm import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
#import torchinfo
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score
)
import torchvision.datasets as D
import multiprocessing
from collections import Counter

## Reproducibility and Device Setup

To ensure consistent results across runs, we fix all random seeds:

* Python random
* NumPy
* PyTorch (CPU and GPU)


### Why this matters

Deep learning training is stochastic. Without fixed seeds:

* results may vary across runs
* debugging becomes difficult
* experiments are not comparable


In [ ]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


In [ ]:
seed_everything()

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")


## Data Preprocessing Pipeline

Since ResNet18 was trained on ImageNet, we must match its expected input format:

### Step 1: Resize

```python
Resize((224, 224))
```
> Convolutional networks are rigidly tied to the spatial hierarchies they were trained on.

### Step 2: Convert grayscale → RGB-like format

```python
Grayscale(num_output_channels=3)
```

### Step 3: Normalize using ImageNet statistics

```python
Normalize(mean=[0.485, 0.456, 0.406],
          std=[0.229, 0.224, 0.225])
```

### Important concept

> Pretrained models expect the same data distribution they were trained on.


In [ ]:
emnist_transforms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    # Use the same distribution the backbone was trained on!
    transforms.Normalize(mean=(0.1307,), std=(0.3081,))
])


In [ ]:
#emnist_trainval_dataset = D.ImageFolder(root=f"/leonardo/pub/userinternal/mcelori1/IntroductionToDeepLearning/Datasets/EMNIST_letters_subset/train", transform=emnist_transforms)
#emnist_test_dataset = D.ImageFolder(root=f"/leonardo/pub/userinternal/mcelori1/IntroductionToDeepLearning/Datasets/EMNIST_letters_subset/test", transform=emnist_transforms)

emnist_trainval_dataset = D.ImageFolder(root=f"data/emnist_letters_subset/train", transform=emnist_transforms)
emnist_test_dataset = D.ImageFolder(root=f"data/emnist_letters_subset/test", transform=emnist_transforms)



## Train / Validation / Test Split

We split the dataset into:

* 80% training
* 20% validation
* separate test set

### Why validation set?

It allows us to:

* detect overfitting
* tune hyperparameters
* decide when to stop training


In [ ]:
# 80/20 train/val split logic
total_size = len(emnist_trainval_dataset)
train_size = int(0.8 * total_size)
val_size = total_size - train_size

emnist_train_dataset, emnist_val_dataset = random_split(
    emnist_trainval_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42)
)

print(f"Train samples: {len(emnist_train_dataset)}")
print(f"Validation samples: {len(emnist_val_dataset)}")
print(f"Test samples: {len(emnist_test_dataset)}")


## Inspect the training dataset

In [ ]:
all_labels = emnist_trainval_dataset.targets
train_indices = emnist_train_dataset.indices
train_labels = [all_labels[i] for i in train_indices]
class_counts = Counter(train_labels)
classes = emnist_trainval_dataset.classes
counts = [class_counts[i] for i in range(len(classes))]

plt.figure(figsize=(12, 5))
plt.bar(range(len(classes)), counts, color='#3498db')
plt.xticks(range(len(classes)), classes, rotation=90)
plt.xlabel("Character")
plt.ylabel("Number of images")
plt.title("EMNIST Letters: Training Dataset Class Distribution")
plt.tight_layout()
plt.show()

In [ ]:
def visualize_dataset(dataset, num_samples=8):
    mean = np.array([0.1307])
    std = np.array([0.3081])
    fig, axes = plt.subplots(1, num_samples, figsize=(num_samples * 2, 3))
    total_samples = len(dataset)
    random_indices = random.sample(range(total_samples), num_samples)
    if hasattr(dataset, "dataset"):
        classes = dataset.dataset.classes
    else:
        classes = dataset.classes
    for i, idx in enumerate(random_indices):
        image_tensor, label_idx = dataset[idx]
        img = image_tensor.squeeze().numpy()
        img = std * img + mean
        img = np.clip(img, 0, 1)
        class_letter = classes[label_idx]
        axes[i].imshow(img, cmap="gray")
        axes[i].set_title(f"Label: {class_letter}", fontsize=11, fontweight='bold')
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
visualize_dataset(emnist_train_dataset, num_samples=8)


## Dataloaders


In [ ]:
batch_size = 128
pin_memory = (device.type == "cuda")
num_workers = min(4, multiprocessing.cpu_count()) if (device.type == "cuda") else 0
persistent_workers = (num_workers > 0)

loader_args = {"batch_size": batch_size, 
               "num_workers": num_workers, 
               "pin_memory": pin_memory,
               "persistent_workers": persistent_workers
              }

emnist_train_loader = DataLoader(emnist_train_dataset, shuffle=True, **loader_args)
emnist_val_loader   = DataLoader(emnist_val_dataset, shuffle=False, **loader_args)
emnist_test_loader  = DataLoader(emnist_test_dataset, shuffle=False, **loader_args)


## Convolutional Neural Network Architecture

We define a compact convolutional neural network for digit classification.

The architecture is divided into two components:

## Feature Extractor

The convolutional layers learn hierarchical visual representations such as:
- edges,
- strokes,
- corners,
- and higher-level digit structures.

Pooling layers progressively reduce spatial dimensionality while preserving important features.

## Classifier

The fully connected layers map learned visual representations to class probabilities.

This separation between:
- feature extraction,
- and classification

is especially important in transfer learning workflows.


## Visualizing the Spatial Transformations

To help your students intuitively track how spatial resolution shrinks while feature depth expands, here is a visual reference you can drop directly into your architectural summary markdown section:

| Pipeline Stage | Layer Type | Output Activation Shape ($C \times H \times W$) | Rationale |
| --- | --- | --- | --- |
| **Input** | Raw Preprocessed Image | $1 \times 28 \times 28$ | Single-channel grayscale MNIST sample. |
| **Stage 1 (Conv)** | `nn.Conv2d(1, 32, k=3, p=1)` | $32 \times 28 \times 28$ | Padding preserves spatial boundaries; channels expand to 32. |
| **Stage 1 (Pool)** | `nn.MaxPool2d(2)` | $32 \times 14 \times 14$ | $2\times2$ pooling downsamples spatial height and width by half. |
| **Stage 2 (Conv)** | `nn.Conv2d(32, 64, k=3, p=1)` | $64 \times 14 \times 14$ | Features deepen to collect more abstract structural shapes. |
| **Stage 2 (Pool)** | `nn.MaxPool2d(2)` | $64 \times 7 \times 7$ | Final spatial reduction. Features are now ready for flattening. |
| **Classifier** | `nn.Flatten()` | $3136$ vector elements | Total inputs passed directly to the first dense layer ($64 \times 7 \times 7$). |


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(inplace=True),
            # Regularizes dense classifier layers
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


## Load the pretrained model

We now initialize a new model and load the pretrained MNIST weights.

However, the output layer must change:
- MNIST has 10 classes
- EMNIST has 26 classes

Therefore:
- we keep the learned convolutional features
- we replace the final classifier layer

This is one of the key ideas in transfer learning.


In [ ]:
model = SimpleCNN(num_classes=26)
model = model.to(device)


In [ ]:
print(model)


Here is the parameter breakdown for your `SimpleCNN`:

| Layer | Type | Calculation | Parameters |
| --- | --- | --- | --- |
| `features.0` | Conv2d | (3 x 3 x 1 x 32) + 32 | 320 |
| `features.1` | BatchNorm2d | 2 x 32 | 64 |
| `features.4` | Conv2d | (3 x 3 x 32 x 64) + 64 | 18,496 |
| `features.5` | BatchNorm2d | 2 x 64 | 128 |
| `classifier.1` | Linear | (64 x 7 x 7 x 128) + 128 | 401,536 |
| `classifier.4` | Linear | (128 x 26) + 26 | 3354 |
| **Total** |  |  | **423,898** |



In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable



In [ ]:
tot, tra = count_parameters(model)
print(f"\nSimpleCNN | total params: {tot}\ttrainable params: {tra}")


In [ ]:
checkpoint_dir = Path("./checkpoints")
checkpoint_path = checkpoint_dir / "mnist_features.pth"

model.features.load_state_dict(torch.load(checkpoint_path))


## Visualizing Learned Convolutional Features

One of the key ideas behind transfer learning is that early convolutional layers learn reusable visual patterns.

These filters often detect:

* edges,
* orientations,
* intensity gradients,
* and simple textures.

Because these low-level visual structures appear in many image datasets, the learned features can transfer effectively to new tasks.


In [ ]:
conv_layer = model.features[0]

filters = conv_layer.weight.detach().cpu()

num_filters = min(16, filters.size(0))
in_channels = filters.size(1)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))

for i, ax in enumerate(axes.flat):
    if i >= num_filters:
        ax.axis("off")
        continue

    filt = filters[i]

    # Per-filter normalization
    f_min, f_max = filt.min(), filt.max()
    if f_max > f_min:
        filt = (filt - f_min) / (f_max - f_min)

    # Shape [1, H, W] -> Permuted to standard [H, W, 1]
    filt_img = filt.permute(1, 2, 0).numpy()
    ax.imshow(filt_img)
    ax.set_title(f"Filter {i+1}", fontsize=10)
    ax.axis("off")

model_name = "SimpleCNN"
plt.suptitle(f"{model_name} First Convolutional Filters", fontsize=14, fontweight="bold", y=0.98)
plt.tight_layout()
plt.show()


## Freeze the feature extractor

Initially, we freeze the convolutional backbone.

This means:
- pretrained features are preserved
- only the classifier is trained

This strategy is useful because:
- pretrained features are already meaningful
- fewer parameters must be optimized
- training becomes more stable


In [ ]:
# Freeze convolutional feature extractor
for param in model.features.parameters():
    param.requires_grad = False

# Train only classifier first
for param in model.classifier.parameters():
    param.requires_grad = True

tot, tra = count_parameters(model)
print(f"\nSimpleCNN | total params: {tot}\ttrainable params: {tra}")


## Early Stopping

We stop training when validation performance stops improving.

### Why?

To prevent:

* overfitting
* unnecessary computation

We restore the best model based on validation loss.


In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=1e-4):
        self.patience = patience
        self.counter = 0
        self.best_loss = float("inf")
        self.best_acc = -float("inf")
        self.best_weights = None
        self.best_epoch = 0
        self.min_delta = min_delta 

    def step(self, model, val_loss, val_acc, epoch):

        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.best_acc = val_acc
            self.best_epoch = epoch+1
            self.counter = 0
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False

        self.counter += 1
        print(
            f"[EarlyStopping] Epoch {epoch+1 if epoch is not None else ''}: "
            f"No improvement → counter {self.counter}/{self.patience}"
        )
        return self.counter >= self.patience

    def restore(self, model):
        model.load_state_dict(self.best_weights)


## Evaluation function

We define a reusable evaluation function.

This function computes:
- loss
- accuracy

on validation or test datasets.

Notice that:
- gradients are disabled with `torch.no_grad()`
- the model is switched to evaluation mode with `model.eval()`


In [ ]:
def evaluate(model, loader, criterion, device):

    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for x, y in loader:

            x = x.to(device)
            y = y.to(device)

            logits = model(x)

            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)

            predictions = logits.argmax(dim=1)

            correct += (predictions == y).sum().item()

            total += y.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy


## Transfer learning on EMNIST

We now train the model on EMNIST.

Training occurs in two stages.

### Stage 1 — Feature Extraction
Only the classifier is trained.

The pretrained convolutional layers remain frozen.

### Stage 2 — Fine-Tuning
After several epochs, we unfreeze the backbone.

This allows the pretrained features to adapt slightly to the new task.

We also use:
- a smaller learning rate for pretrained layers
- a larger learning rate for the new classifier

This is standard practice in transfer learning.

> **In larger networks, we often unfreeze only later convolutional blocks because early layers contain highly generic visual features**.

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=1e-3
)

early_stopping = EarlyStopping(patience=3)

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

print("TRANSFER LEARNING ON EMNIST")

epochs = 30

for epoch in range(epochs):

    # Fine-tuning phase
    if epoch == 3:

        print("\nUnfreezing convolutional backbone...\n")

        for param in model.features.parameters():
            param.requires_grad = True

        optimizer = torch.optim.Adam([
            {
                "params": model.classifier.parameters(),
                "lr": 1e-4
            },
            {
                "params": model.features.parameters(),
                "lr": 1e-5
            }
        ])

    # Training
    model.train()

    total_train_loss = 0
    correct = 0
    total = 0

    for x, y in tqdm(emnist_train_loader):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item() * x.size(0)

        predictions = logits.argmax(dim=1)

        correct += (predictions == y).sum().item()

        total += y.size(0)

    train_loss = total_train_loss / total

    train_accuracy = correct / total

    # Validation
    val_loss, val_accuracy = evaluate(
        model,
        emnist_val_loader,
        criterion, device
    )

    # Store metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    # Epoch summary
    print(
        f"Epoch {epoch+1:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

    # Early stopping
    if early_stopping.step(model, val_loss, val_accuracy, epoch):
        print("\nEarly stopping triggered.")
        break


# Restore best model
early_stopping.restore(model)

print("\n================================================")
print("BEST MODEL SUMMARY")
print("================================================")
print(f"Best Epoch         : {early_stopping.best_epoch}")
print(f"Best Validation Acc: {early_stopping.best_acc:.4f}")


## Visualize Training Curves

We plot:
- training loss
- validation loss
- training accuracy
- validation accuracy

These plots help diagnose:
- overfitting
- underfitting
- convergence behavior


In [ ]:
epochs_range = range(1, len(train_losses) + 1)

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

plt.plot(epochs_range, train_losses, label="Training Loss", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_losses, label="Validation Loss", color="#d95f02", linewidth=2.5, linestyle="--")

best_epoch = early_stopping.best_epoch
best_loss = early_stopping.best_loss
plt.scatter(best_epoch, best_loss, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

plt.title("Training vs Validation Loss", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Cross Entropy Loss", fontsize=11, labelpad=10)

plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

plt.legend(loc="upper right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
epochs_range = range(1, len(train_losses) + 1)
best_epoch = early_stopping.best_epoch
best_acc = early_stopping.best_acc

plt.style.use('seaborn-v0_8-whitegrid') 
plt.figure(figsize=(10, 5), dpi=100)

plt.plot(epochs_range, train_accuracies, label="Training Accuracy", color="#2b5c8f", linewidth=2.5)
plt.plot(epochs_range, val_accuracies, label="Validation Accuracy", color="#d95f02", linewidth=2.5, linestyle="--")

plt.scatter(best_epoch, best_acc, color="#d95f02", edgecolor="black", 
            s=100, zorder=5, label=f"Best Model (Epoch {best_epoch})")

plt.title("Training vs Validation Accuracy", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Training Epochs", fontsize=11, labelpad=10)
plt.ylabel("Accuracy", fontsize=11, labelpad=10)

plt.grid(True, linestyle=":", alpha=0.6, color="#cccccc")

plt.legend(loc="lower right", frameon=True, facecolor="white", edgecolor="#e0e0e0", fontsize=10)

plt.tight_layout()
plt.show()


## Final test evaluation

After training, we evaluate the final model on the test set.

We compute:
- accuracy
- macro F1 score
- weighted F1 score

Why use F1 scores?

Accuracy alone can sometimes hide poor class-specific performance.

F1 scores provide a more balanced evaluation.


In [ ]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for x, y in emnist_test_loader:

        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        predictions = logits.argmax(dim=1)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_labels.extend(
            y.cpu().numpy()
        )

accuracy = np.mean(
    np.array(all_predictions) == np.array(all_labels)
)

f1_macro = f1_score(
    all_labels,
    all_predictions,
    average="macro"
)

f1_weighted = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

print("\n================================================")
print("FINAL TEST METRICS")
print("================================================")

print(f"Accuracy      : {accuracy:.4f}")
print(f"Macro F1 Score: {f1_macro:.4f}")
print(f"Weighted F1   : {f1_weighted:.4f}")


##  Confusion matrix

The confusion matrix shows:
- correct predictions
- systematic mistakes between classes

This visualization is especially useful for handwritten character recognition because some letters look visually similar.

Examples:
- O vs Q
- I vs L
- C vs G


In [ ]:
class_names = [
    chr(i)
    for i in range(ord("A"), ord("Z") + 1)
]

cm = confusion_matrix(
    all_labels,
    all_predictions
)

fig, ax = plt.subplots(figsize=(10, 10))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

disp.plot(
    cmap="Blues",
    ax=ax,
    colorbar=False
)

ax.grid(False)

ax.set_xticklabels(class_names, rotation=0)

plt.title("EMNIST Letters Confusion Matrix")
plt.show()


## Classification report

Finally, we print a detailed classification report containing:
- precision
- recall
- F1 score

for each class individually.

This helps identify:
- strong classes
- weak classes
- difficult character pairs


In [ ]:
print("\n================================================")
print("CLASSIFICATION REPORT")
print("================================================")

print(classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
))

## Key takeaways

This notebook demonstrates the full transfer learning pipeline:

1. Train on a source task
2. Save learned features
3. Reuse pretrained representations
4. Replace task-specific layers
5. Freeze and fine-tune selectively
6. Evaluate generalization performance

Most modern deep learning systems use this exact workflow at much larger scale.

Examples include:
- ImageNet pretrained vision models
- pretrained language models
- speech recognition systems
- foundation models